In [1]:
import ray
import time


# A regular Python function.
def normal_function():
    return 1


# By adding the `@ray.remote` decorator, a regular Python function
# becomes a Ray remote function.
@ray.remote
def my_function():
    return 1


# To invoke this remote function, use the `remote` method.
# This will immediately return an object ref (a future) and then create
# a task that will be executed on a worker process.
obj_ref = my_function.remote()

# The result can be retrieved with ``ray.get``.
assert ray.get(obj_ref) == 1


@ray.remote
def slow_function():
    time.sleep(10)
    return 1


# Ray tasks are executed in parallel.
# All computation is performed in the background, driven by Ray's internal event loop.
for _ in range(4):
    # This doesn't block.
    slow_function.remote()

/Users/yinyajun/miniconda3/envs/tomo/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
2026-02-27 10:22:56,372	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/Users/yinyajun/miniconda3/envs/tomo/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [2]:
# Specify required resources.
# @ray.remote(num_cpus=4, num_gpus=2)
@ray.remote(num_cpus=4)
def my_function():
    return 1


# Override the default resource requirements.
r = my_function.options(num_cpus=2).remote()
r

ObjectRef(f4402ec78d3a2607ffffffffffffffffffffffff0100000001000000)

In [3]:
ray.get(r)

1

In [4]:
!ray summary tasks

/Users/yinyajun/miniconda3/envs/tomo/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(

======== Tasks Summary: 2026-02-27 10:23:12.439052 ========
Stats:
------------------------------------
total_actor_scheduled: 0
total_actor_tasks: 0
total_tasks: 6


Table (group by func_name):
------------------------------------
    FUNC_OR_CLASS_NAME    STATE_COUNTS    TYPE
0   slow_function         FINISHED: 4     NORMAL_TASK
1   my_function           FINISHED: 2     NORMAL_TASK



In [5]:
!ray job list

/Users/yinyajun/miniconda3/envs/tomo/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
Job submission server address: http://127.0.0.1:8265
[JobDetails(type=<JobType.DRIVER: 'DRIVER'>, job_id='01000000', submission_id=None, driver_info=DriverInfo(id='01000000', node_ip_address='127.0.0.1', pid='2721'), status=<JobStatus.RUNNING: 'RUNNING'>, entrypoint='(interactive_shell) /Users/yinyajun/miniconda3/envs/tomo/bin/python -m ipykernel_launcher -f /var/folders/bx/psqvr3j17f1611pq0ht00k3r0000gn/T/kernel-0d9d395f-1862-4d47-852b-90e7df84475910166452488974750055/connection.json --IPCompleter.use_jedi=False --IPKernelApp.log_level=INFO', message=None, error_type=None, start_time=1772158976876, end_time=0, metadata={}, runtime_env={}, driver_agent_http_address=None, driver_node_id=None, driver_exit_code=None)]


In [6]:
!ray job stop 01000000

/Users/yinyajun/miniconda3/envs/tomo/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
Job submission server address: http://127.0.0.1:8265
Attempting to stop job '01000000'
Traceback (most recent call last):
  File "/Users/yinyajun/miniconda3/envs/tomo/bin/ray", line 6, in <module>
    sys.exit(main())
             ^^^^^^
  File "/Users/yinyajun/miniconda3/envs/tomo/lib/python3.12/site-packages/ray/scripts/scripts.py", line 2894, in main
    return cli()
           ^^^^^
  File "/Users/yinyajun/miniconda3/envs/tomo/lib/python3.12/site-packages/click/core.py", line 1485, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yinyajun/miniconda3/envs/tomo/lib/python3.12/site-packages/click/core.py", line 1406, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/Users/yinyaj

In [7]:
@ray.remote
def function_with_an_argument(value):
    return value + 1


obj_ref1 = my_function.remote()
assert ray.get(obj_ref1) == 1

# You can pass an object ref as an argument to another Ray task.
obj_ref2 = function_with_an_argument.remote(obj_ref1)
assert ray.get(obj_ref2) == 2

In [8]:
object_refs = [slow_function.remote() for _ in range(2)]
# Return as soon as one of the tasks finished execution.
ready_refs, remaining_refs = ray.wait(object_refs, num_returns=1, timeout=None)

In [9]:
ready_refs

[ObjectRef(80e22aed7718a125ffffffffffffffffffffffff0100000001000000)]

In [10]:
remaining_refs

[ObjectRef(8849b62d89cb30f9ffffffffffffffffffffffff0100000001000000)]

In [11]:
# By default, a Ray task only returns a single Object Ref.
@ray.remote
def return_single():
    return 0, 1, 2


object_ref = return_single.remote()
assert ray.get(object_ref) == (0, 1, 2)


# However, you can configure Ray tasks to return multiple Object Refs.
@ray.remote(num_returns=3)
def return_multiple():
    return 0, 1, 2


object_ref0, object_ref1, object_ref2 = return_multiple.remote()
assert ray.get(object_ref0) == 0
assert ray.get(object_ref1) == 1
assert ray.get(object_ref2) == 2

In [12]:
@ray.remote(num_returns=3)
def return_multiple_as_generator():
    for i in range(3):
        yield i


# NOTE: Similar to normal functions, these objects will not be available
# until the full task is complete and all returns have been generated.
a, b, c = return_multiple_as_generator.remote()

In [13]:
a

ObjectRef(85748392bcd969ccffffffffffffffffffffffff0100000001000000)

In [14]:
b

ObjectRef(85748392bcd969ccffffffffffffffffffffffff0100000002000000)

In [15]:
c

ObjectRef(85748392bcd969ccffffffffffffffffffffffff0100000003000000)

In [16]:
ray.get([a,b,c])

[0, 1, 2]

In [20]:
@ray.remote
def blocking_operation():
    time.sleep(10e6)


obj_ref = blocking_operation.remote()
ray.cancel(obj_ref)

try:
    ray.get(obj_ref)
except ray.exceptions.TaskCancelledError:
    print("Object reference was cancelled.")

Object reference was cancelled.


In [21]:
import ray


@ray.remote
def f():
    return 1


@ray.remote
def g():
    # Call f 4 times and return the resulting object refs.
    return [f.remote() for _ in range(4)]


@ray.remote
def h():
    # Call f 4 times, block until those 4 tasks finish,
    # retrieve the results, and return the values.
    return ray.get([f.remote() for _ in range(4)])



In [22]:
ray.get(g.remote())

[ObjectRef(2bb9719904ecc03effffffffffffffffffffffff0100000001000000),
 ObjectRef(38d76a17b38aee46ffffffffffffffffffffffff0100000001000000),
 ObjectRef(2cdbb5204a93cb1dffffffffffffffffffffffff0100000001000000),
 ObjectRef(360884a1394ee07fffffffffffffffffffffffff0100000001000000)]

In [23]:
ray.get(h.remote())

[1, 1, 1, 1]